In [1]:
from openai import OpenAI
# Configured by environment variables
client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_today_weather",
            "description": "Get a description of today's weather",
            "parameters": {},
            "returns": {
                "type": "string",
                "description": "Return string containing the weather description, and the temperature."
            }
        }
    },
]

content = [
    {
        "type": "text",
        "text": "What is the weather like today?",
    }
]

messages = [
    {
        "role": "user",
        "content": content,
    }
]

In [3]:
chat_response = client.chat.completions.create(
  model="Ternary-Bonsai-27B-Q2_0.gguf",
  messages=messages,
  tools=tools,
  max_tokens=4096,
  temperature=0.7,
  top_p=0.95,
  extra_body={
      "top_k": 20,
      # "enable_thinking": False,
  },
)

In [4]:
response_message = chat_response.choices[0].message
if response_message.tool_calls:
    # 1. Append the assistant's tool-call message to the conversation
    messages.append(response_message)

    for tool_call in response_message.tool_calls:
        if tool_call.function.name == "get_today_weather":
            # 2. Actually run your function here
            result = "Sunny, 24°C"  # replace with your real implementation

            print('Found tool!')
            # 3. Append the tool result, matching tool_call_id
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

Found tool!


In [5]:
final_response = client.chat.completions.create(
    model="Ternary-Bonsai-27B-Q2_0.gguf",
    messages=messages,
    tools=tools,
    max_tokens=4096,
    temperature=0.7,
    top_p=0.95,
    extra_body={"top_k": 20},
)

print(final_response.choices[0].message.content)

Today's weather is sunny with a temperature of 24°C.
